# Atividade de Entrega — Aula 10: pipeline de limpeza de exames laboratoriais

**Análise e Ciência de Dados** (Especialização, IFRN) — Unidade 2, Aula 10

**Nome:** ALEXANDRE CLEMENTE

**O que entregar (Tarefa no Moodle, prazo 30/10/2026):**
1. este notebook **executado e comentado** (`.ipynb`), com o seu nome no arquivo;
2. o dataset limpo `exames_laboratoriais_limpo.csv`;
3. (opcional) o dicionário de dados em CSV, se você não o deixou no próprio notebook.

**Dados:** `dados/exames_laboratoriais.csv` — 155 registros **sintéticos** de exames laboratoriais (nenhum paciente real).

**Regras de documentação:** antes de cada bloco de código, escreva uma célula de texto explicando **o que** será feito e **por quê**; depois, comente o resultado (o que mudou? quantas linhas/valores foram afetados?). Use os notebooks da aula como referência.

## Requisitos

| # | Etapa | O que fazer |
|---|---|---|
| 1 | Extract e diagnóstico | carregar, `shape`, `dtypes`, ausentes, duplicatas e `value_counts` das categorias |
| 2 | Duplicatas | remover as linhas repetidas e conferir que `id_exame` ficou único |
| 3 | Tipos | `data_coleta` (dois formatos) → `datetime`; `resultado` (`"84,2"`) → `float` |
| 4 | Padronização | `sexo` → `F`/`M`; `prioridade` → `Rotina`/`Urgente`/`Emergência`; `laboratorio` → nomes oficiais sem espaços extras |
| 5 | Valores inválidos e ausentes | o código **999** em `resultado` significa "amostra não processada" → ausente; imputar `resultado` pela mediana **do mesmo tipo de exame** e `idade` pela mediana |
| 6 | Engenharia de atributos | `faixa_etaria` com `pd.cut` (18-39, 40-59, 60-79, 80+); `mes_coleta`; `alterado` (True/False) usando a tabela de referência abaixo |
| 7 | Codificação | `prioridade_cod` ordinal (Rotina=0, Urgente=1, Emergência=2); one-hot de `tipo_exame` |
| 8 | Validação | pelo menos 3 `assert` (ex.: sem ausentes, `id_exame` único, resultados positivos) |
| 9 | Load | exportar `exames_laboratoriais_limpo.csv` (sem índice) e mostrar o **dicionário de dados** (coluna, tipo, descrição) |
| 10 | Conclusão | 3 a 5 linhas: problemas encontrados, decisões tomadas e um achado com os dados limpos |

**Valores de referência (simplificados, apenas para fins didáticos):**

| Exame | Unidade | Faixa considerada normal |
|---|---|---|
| Glicemia | mg/dL | 70 a 99 |
| Colesterol total | mg/dL | abaixo de 190 |
| Creatinina | mg/dL | 0,6 a 1,2 |
| Hemoglobina | g/dL | 12,0 a 17,5 |

Nomes oficiais dos laboratórios: `LabNatal`, `LabSaúde RN`, `Laboratório Central`.

**Pontos de controle (para você conferir):** 150 linhas após remover duplicatas · 4 resultados com código 999 e 6 idades ausentes (após remover duplicatas) · nenhum ausente no dataset final · seguindo exatamente os requisitos, o arquivo final tem 150 linhas e 18 colunas.

**Critérios de avaliação (10 pontos):** limpeza e tipos corretos (3,0) · engenharia de atributos e codificação (2,5) · validação e exportação do dataset (1,5) · documentação — células de texto, comentários e dicionário de dados (2,0) · conclusão (1,0).

## 1. Extract e diagnóstico

Carregar o dataset bruto e fazer um diagnóstico geral: dimensões, tipos, ausentes, duplicatas e frequência das categorias.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# criar a pasta onde está o dataset, e onde ficara a saida com dataset limpo
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 14)
DADOS = Path("dados")
SAIDA = Path("saida")
SAIDA.mkdir(exist_ok=True)

# 1. Carregamento dos dados brutos
exames_bruto = pd.read_csv(DADOS / "exames_laboratoriais.csv", encoding="utf-8")

print("==================================================")
print("1. DIAGNÓSTICO GERAL DA BASE")
print("==================================================")
print(f"- Dimensões (Linhas, Colunas): {exames_bruto.shape}")
print(f"- Linhas inteiramente duplicadas: {exames_bruto.duplicated().sum()}")

print("\n--------------------------------------------------")
print("2. RESUMO DE TIPOS, VALORES AUSENTES, PERCENTUAL DE AUSENTES, CARDINALIDADE")
print("--------------------------------------------------")
df_diagnostico = pd.DataFrame({
    "tipo": exames_bruto.dtypes.astype(str),
    "ausentes": exames_bruto.isna().sum(),
    "percentual (%)": (exames_bruto.isna().sum() / len(exames_bruto) * 100).round(1),
    "Nununique": (exames_bruto.nunique())
})
print(df_diagnostico)

print("\n--------------------------------------------------")
print("3. VALORES SENTINELA E OUTLIERS")
print("--------------------------------------------------")
# Checando o código de erro 999 em 'resultado'
qtd_999 = (exames_bruto["resultado"] == "999").sum()
print(f"- Código '999' (amostra não processada) na coluna 'resultado': {qtd_999}")

# Estatísticas da coluna 'idade'
print("\nEstatísticas da Idade:")
print(exames_bruto["idade"].describe().round(1))

print("\n--------------------------------------------------")
print("4. FREQUÊNCIA DAS VARIÁVEIS CATEGÓRICAS")
print("--------------------------------------------------")
for col in ["sexo", "tipo_exame", "unidade", "laboratorio", "prioridade"]:
    print(f"\n{col.upper()}:")
    print(exames_bruto[col].value_counts(dropna=False).to_dict())

print("\nAmostra dos 6 primeiros registros:\n")
print(exames_bruto.head(6).to_string())
print("\n---")


1. DIAGNÓSTICO GERAL DA BASE
- Dimensões (Linhas, Colunas): (155, 10)
- Linhas inteiramente duplicadas: 5

--------------------------------------------------
2. RESUMO DE TIPOS, VALORES AUSENTES, PERCENTUAL DE AUSENTES, CARDINALIDADE
--------------------------------------------------
                tipo  ausentes  percentual (%)  Nununique
id_exame         str         0             0.0        150
id_paciente      str         0             0.0        106
data_coleta      str         0             0.0         81
sexo             str         0             0.0          4
idade        float64         7             4.5         65
tipo_exame       str         0             0.0          4
resultado        str         0             0.0        134
unidade          str         0             0.0          2
laboratorio      str         0             0.0          5
prioridade       str         0             0.0          6

--------------------------------------------------
3. VALORES SENTINELA E OU

## 2. Duplicatas

Exibir a quantidade a as linhas duplicadas e/ou repetidas. Conferir que `id_exame` ficou único.

In [2]:
print("\n--------------------------------------------------")
print("4. DUPLICATAS")
print("--------------------------------------------------")

print("\nAntes da remoção de duplicatas:", len(exames_bruto))

print("\nRegistros duplicados:")
print(exames_bruto[exames_bruto.duplicated()])

exames_bruto_sem_duplicatas = exames_bruto.drop_duplicates().reset_index(drop=True)

print("\nApós remover duplicatas:", len(exames_bruto_sem_duplicatas), "| id_exame único?", exames_bruto_sem_duplicatas["id_exame"].is_unique)




--------------------------------------------------
4. DUPLICATAS
--------------------------------------------------

Antes da remoção de duplicatas: 155

Registros duplicados:
    id_exame id_paciente data_coleta sexo  idade        tipo_exame resultado unidade          laboratorio prioridade
92   EX-0091    PAC-0153  2026-07-08    F    NaN        Creatinina      0,82   mg/dL             LabNatal     Rotina
100  EX-0088    PAC-0007  2026-07-11    M   26.0          Glicemia     103,4   mg/dL             LabNatal    Urgente
135  EX-0023    PAC-0082  2026-09-23    F   28.0  Colesterol total     226,6   mg/dL            labnatal     Urgente
136  EX-0086    PAC-0030  2026-08-31    M   42.0        Creatinina       999   mg/dL             LabNatal     ROTINA
154  EX-0093    PAC-0072  2026-07-20    F   30.0          Glicemia      72,3   mg/dL  Laboratório Central     Rotina

Após remover duplicatas: 150 | id_exame único? True


## 3. Transformação de tipos

Nota-se no diagnóstico "4. FREQUÊNCIA DAS VARIÁVEIS CATEGÓRICAS" que algumas colunas estão com tipos de dados inadequados para a análise.

Por exemplo, a coluna "data_coleta" possui dois formatos diferentes, e a coluna "resultado" contém valores no formato de string que precisam ser convertidos para números. Os atributos "sexo" e "laboratorio" também precisam ser padronizados (retirar espaços em branco, converter para minúsculas e transformar em valores categóricos únicos) para facilitar a análise. A seguir, realizaremos as transformações necessárias para corrigir os tipos de dados e padronizar os valores.

In [3]:
exames_transformados = exames_bruto_sem_duplicatas.copy()

print("\n--------------------------------------------------")
print(" 5. TRANSFORMAÇÃO DE TIPOS")
print("--------------------------------------------------")

# Transformando a coluna "data_coleta" para o tipo datetime, considerando os dois formatos possíveis: ISO (YYYY-MM-DD) e BR (DD/MM/YYYY)
datas_coletas = exames_transformados["data_coleta"]
iso = pd.to_datetime(datas_coletas, format="%Y-%m-%d", errors="coerce")
br = pd.to_datetime(datas_coletas, format="%d/%m/%Y", errors="coerce")
exames_transformados["data_coleta"] = iso.fillna(br)

print("\nTransformação da coluna 'data_coleta'")
print("No formato ISO:", iso.notna().sum(), "| no formato dd/mm/aaaa:", br.notna().sum())
print("dtype da coluna 'data_coleta':", exames_transformados["data_coleta"].dtype)
print("Datas que ficaram sem converter:", exames_transformados["data_coleta"].isna().sum())

print("\nAmostras transformadas:")
print(exames_transformados[["id_exame", "data_coleta"]].head(2).to_string(index=False))

print("\n--------------------------------------------------")

print("\nTransformação da coluna 'resultado'")

def resultado_para_float(serie):
    """'72,3' -> 72.3 (valores ausentes continuam NaN)."""
    return (serie.str.strip()
                 .str.replace(".", "", regex=False)
                 .str.replace(",", ".", regex=False)
                 .astype(float))

exames_transformados["resultado"] = resultado_para_float(exames_transformados["resultado"])
exames_transformados["resultado"] = pd.to_numeric(exames_transformados["resultado"], errors="coerce", downcast="float").round(2)
print("dtype da coluna 'resultado':", exames_transformados["resultado"].dtype)
print("Valores ausentes na coluna 'resultado':", exames_transformados["resultado"].isna().sum())

print("\nAmostras transformadas:")
print(exames_transformados[["id_exame", "resultado"]].head(2).to_string(index=False))


--------------------------------------------------
 5. TRANSFORMAÇÃO DE TIPOS
--------------------------------------------------

Transformação da coluna 'data_coleta'
No formato ISO: 136 | no formato dd/mm/aaaa: 14
dtype da coluna 'data_coleta': datetime64[us]
Datas que ficaram sem converter: 0

Amostras transformadas:
id_exame data_coleta
 EX-0120  2026-08-14
 EX-0138  2026-08-02

--------------------------------------------------

Transformação da coluna 'resultado'
dtype da coluna 'resultado': float32
Valores ausentes na coluna 'resultado': 0

Amostras transformadas:
id_exame  resultado
 EX-0120 227.199997
 EX-0138  12.500000


## 4. Padronização das categorias

Nota-se no diagnóstico "4. FREQUÊNCIA DAS VARIÁVEIS CATEGÓRICAS" que algumas colunas estão com valores inconsistentes ou não padronizados.
Iremos padronizar os valores das colunas "sexo", "prioridade" e "laboratorio" para garantir consistência nos dados. Isso inclui a remoção de espaços em branco, a conversão para letras maiúsculas ou minúsculas conforme necessário, e a substituição de valores abreviados por seus equivalentes completos.

In [4]:
exames_padronizados = exames_transformados.copy()

print("\nTransformação da coluna 'sexo'")

print("\nANTES:")
print(exames_transformados["sexo"].value_counts(dropna=False).to_dict())

exames_padronizados["sexo"] = exames_padronizados["sexo"].str.strip().str[0].str.upper()

print("\nDEPOIS:")
print(exames_padronizados["sexo"].value_counts(dropna=False).to_dict())

print("\n--------------------------------------------------")

print("\nTransformação da coluna 'prioridade'")

print("\nANTES:")
print(exames_transformados["prioridade"].value_counts(dropna=False).to_dict())

map_prioridade = {"Rotina": "ROTINA", "ROTINA": "ROTINA", "Urgente": "URGENTE", "URGENTE": "URGENTE", "Emergência": "EMERGÊNCIA", "EMERGÊNCIA": "EMERGÊNCIA"}
exames_padronizados["prioridade"] = exames_padronizados["prioridade"].str.strip().map(map_prioridade).str.capitalize()

print("\nDEPOIS:")
print(exames_padronizados["prioridade"].value_counts(dropna=False).to_dict())

print("\n--------------------------------------------------")

print("\nTransformação da coluna 'laboratorio'")

print("\nANTES:")
print(exames_transformados["laboratorio"].value_counts(dropna=False).to_dict())

map_laboratorio = {"LabSaúde RN": "LabSaúde RN", "labsaúde rn": "LabSaúde RN", "Laboratório Central": "Laboratório Central", "LabNatal": "LabNatal", "labnatal": "LabNatal"}
exames_padronizados["laboratorio"] = exames_padronizados["laboratorio"].str.strip().map(map_laboratorio)

print("\nDEPOIS:")
print(exames_padronizados["laboratorio"].value_counts(dropna=False).to_dict())

print("\nAmostra do dataset padronizados:")
print(exames_padronizados.head(6).to_string())


Transformação da coluna 'sexo'

ANTES:
{'M': 70, 'F': 66, 'masculino': 10, 'Feminino': 4}

DEPOIS:
{'M': 80, 'F': 70}

--------------------------------------------------

Transformação da coluna 'prioridade'

ANTES:
{'Rotina': 95, 'Urgente': 34, 'ROTINA': 10, 'Emergência': 9, 'URGENTE': 1, 'EMERGÊNCIA': 1}

DEPOIS:
{'Rotina': 105, 'Urgente': 35, 'Emergência': 10}

--------------------------------------------------

Transformação da coluna 'laboratorio'

ANTES:
{'LabSaúde RN': 53, 'Laboratório Central': 48, 'LabNatal': 39, ' labnatal ': 5, ' labsaúde rn ': 5}

DEPOIS:
{'LabSaúde RN': 58, 'Laboratório Central': 48, 'LabNatal': 44}

Amostra do dataset padronizados:
  id_exame id_paciente data_coleta sexo  idade        tipo_exame   resultado unidade          laboratorio  prioridade
0  EX-0120    PAC-0166  2026-08-14    F   73.0  Colesterol total  227.199997   mg/dL  Laboratório Central      Rotina
1  EX-0138    PAC-0035  2026-08-02    M   19.0       Hemoglobina   12.500000    g/dL  Labora

## 5. Valores inválidos e ausentes

Nota-se que a coluna "resultado" contém valores inválidos, como o código **999**, que indica "amostra não processada". Esses valores devem ser tratados como ausentes. Além disso, algumas idades estão ausentes e precisam ser imputadas. A estratégia adotada será substituir os valores ausentes de "resultado" pela mediana do mesmo tipo de exame e os valores ausentes de "idade" pela mediana geral.

In [5]:
exames_padronizados_2 = exames_padronizados.copy()

print("\nPadronização das colunas 'resultado' e 'idade'")

print("\nANTES:")
print("Valores ausentes ou '999' em 'resultado':", exames_padronizados[exames_padronizados["resultado"].isna() | exames_padronizados["resultado"] == 999].shape[0])
print("Valores ausentes em 'idade':", exames_padronizados[exames_padronizados["idade"].isna()].shape[0])

exames_padronizados_2["resultado"] = exames_padronizados["resultado"].replace(999, np.nan)
exames_padronizados_2["idade"] = exames_padronizados["idade"].fillna(exames_padronizados["idade"].median())

medianas_por_exame = (
    exames_padronizados_2.groupby("tipo_exame")["resultado"]
    .transform("median")
)

exames_padronizados_2["resultado"] = (
    exames_padronizados_2["resultado"].fillna(medianas_por_exame)
)

print("\nDEPOIS:")
print("Valores ausentes ou '999' em 'resultado':", exames_padronizados_2[exames_padronizados_2["resultado"].isna() | exames_padronizados_2["resultado"] == 999].shape[0])
print("Valores ausentes em 'idade':", exames_padronizados_2[exames_padronizados_2["idade"].isna()].shape[0])


Padronização das colunas 'resultado' e 'idade'

ANTES:
Valores ausentes ou '999' em 'resultado': 0
Valores ausentes em 'idade': 6

DEPOIS:
Valores ausentes ou '999' em 'resultado': 0
Valores ausentes em 'idade': 0


## 6. Engenharia de atributos

Será criada a coluna `faixa_etaria` utilizando `pd.cut` para categorizar as idades em faixas (18-39, 40-59, 60-79, 80+). Além disso, será criada a coluna `mes_coleta` para indicar o mês de coleta do exame e a coluna `alterado`, que indicará se o resultado do exame está fora da faixa de referência estabelecida na tabela de referência.

In [6]:
limites = [0, 18, 40, 60, 80, 120]
rotulos = ["0-17", "18-39", "40-59", "60-79", "80+"]
exames_padronizados_2["faixa_etaria"] = pd.cut(exames_padronizados_2["idade"], bins=limites, labels=rotulos, right=False)

print("\nDistribuição das faixas etárias:")
print(exames_padronizados_2["faixa_etaria"].value_counts().sort_index())

exames_padronizados_2["mes_coleta"] = exames_padronizados_2["data_coleta"].dt.month

print("\nDistribuição dos meses de coleta:")
print(exames_padronizados_2["mes_coleta"].value_counts().sort_index())

# Definir os limites de referência por tipo de exame
referencias = {
    "Glicemia": (70.0, 99.0),
    "Colesterol total": (-np.inf, 190.0),
    "Creatinina": (0.6, 1.2),
    "Hemoglobina": (12.0, 17.5),
}

limites = exames_padronizados_2["tipo_exame"].map(referencias)
referencias_df = pd.DataFrame(
    limites.tolist(),
    columns=["minimo", "maximo"],
    index=exames_padronizados_2.index,
)

if referencias_df.isna().any().any():
    raise ValueError("Há tipos de exame sem referência cadastrada.")

exames_padronizados_2["alterado"] = (
    (exames_padronizados_2["resultado"] < referencias_df["minimo"])
    | (exames_padronizados_2["resultado"] > referencias_df["maximo"])
)

print("\nAmostra do dataset de algumas colunas:")

colunas_novas = ["id_exame", "prioridade", "faixa_etaria", "mes_coleta", "alterado"]
print(exames_padronizados_2[colunas_novas].head(6).to_string())



Distribuição das faixas etárias:
faixa_etaria
0-17      0
18-39    47
40-59    41
60-79    44
80+      18
Name: count, dtype: int64

Distribuição dos meses de coleta:
mes_coleta
7    44
8    60
9    46
Name: count, dtype: int64

Amostra do dataset de algumas colunas:
  id_exame  prioridade faixa_etaria  mes_coleta  alterado
0  EX-0120      Rotina        60-79           8      True
1  EX-0138      Rotina        18-39           8     False
2  EX-0127      Rotina        60-79           8     False
3  EX-0066      Rotina        18-39           9     False
4  EX-0089     Urgente        18-39           9     False
5  EX-0010  Emergência        40-59           8      True


## 7. Codificação das categorias

Será criado a coluna `prioridade_cod` em codificação ordinal (Rotina=0, Urgente=1, Emergência=2) e será realizada a codificação one-hot da coluna `tipo_exame`. Assim , teremos colunas binárias indicando a presença de cada tipo de exame que facilitará a análise e modelagem dos dados.

In [7]:
exames_laboratoriais_limpo = exames_padronizados_2.copy()

# Criar a coluna `prioridade_cod` em codificação ordinal (Rotina=0, Urgente=1, Emergência=2)
prioridade_map = {"Rotina": 0, "Urgente": 1, "Emergência": 2}
exames_laboratoriais_limpo["prioridade_cod"] = exames_padronizados_2["prioridade"].map(prioridade_map)

# Realizar a codificação one-hot da coluna `tipo_exame`
# Gerar as colunas one-hot apenas para a série tipo_exame
one_hot_exame = pd.get_dummies(exames_laboratoriais_limpo["tipo_exame"], prefix="tipo_exame", dtype=int)

# Concatenar o DataFrame original com as novas colunas
exames_laboratoriais_limpo = pd.concat([exames_laboratoriais_limpo, one_hot_exame], axis=1)

# Padronizando os nomes das colunas para minúsculas e substituindo espaços por underscores
exames_laboratoriais_limpo.columns = exames_laboratoriais_limpo.columns.str.lower().str.replace(" ", "_")

print("\nAmostra do dataset de algumas colunas:")

colunas_novas = [
    "id_exame",
    "prioridade",
    "prioridade_cod",
    "tipo_exame",
    "tipo_exame_colesterol_total",
    "tipo_exame_creatinina",
    "tipo_exame_glicemia",
    "tipo_exame_hemoglobina"
]
print(exames_laboratoriais_limpo[colunas_novas].head(6).to_string())


Amostra do dataset de algumas colunas:
  id_exame  prioridade  prioridade_cod        tipo_exame  tipo_exame_colesterol_total  tipo_exame_creatinina  tipo_exame_glicemia  tipo_exame_hemoglobina
0  EX-0120      Rotina               0  Colesterol total                            1                      0                    0                       0
1  EX-0138      Rotina               0       Hemoglobina                            0                      0                    0                       1
2  EX-0127      Rotina               0          Glicemia                            0                      0                    1                       0
3  EX-0066      Rotina               0        Creatinina                            0                      1                    0                       0
4  EX-0089     Urgente               1        Creatinina                            0                      1                    0                       0
5  EX-0010  Emergência              

## 8. Validação

Fazer pelo menos 3 `assert` para validar o dataset final (ex.: sem ausentes, `id_exame` único, resultados positivos).

In [8]:
assert exames_laboratoriais_limpo.isnull().sum().sum() == 0, "O dataset final possui valores ausentes."
assert exames_laboratoriais_limpo['id_exame'].nunique() == exames_laboratoriais_limpo.shape[0], "A coluna `id_exame` não é única."
assert (exames_laboratoriais_limpo["resultado"] > 0).all(), ("A coluna `resultado` deve conter apenas valores positivos.")

# exibir a validação do dataset final
print("\nValidação do dataset final:")
print("- Sem valores ausentes:", exames_laboratoriais_limpo.isnull().sum().sum() == 0)
print("- `id_exame` único:", exames_laboratoriais_limpo['id_exame'].nunique() == exames_laboratoriais_limpo.shape[0])
print("- Resultados positivos:", (exames_laboratoriais_limpo["resultado"] > 0).all())

# Exibir as primeiras linhas do dataset final com codificação
print("\nAmostra do dataset final com codificação:")
print(exames_laboratoriais_limpo.head(6).to_string())


Validação do dataset final:
- Sem valores ausentes: True
- `id_exame` único: True
- Resultados positivos: True

Amostra do dataset final com codificação:
  id_exame id_paciente data_coleta sexo  idade        tipo_exame   resultado unidade          laboratorio  prioridade faixa_etaria  mes_coleta  alterado  prioridade_cod  tipo_exame_colesterol_total  tipo_exame_creatinina  tipo_exame_glicemia  tipo_exame_hemoglobina
0  EX-0120    PAC-0166  2026-08-14    F   73.0  Colesterol total  227.199997   mg/dL  Laboratório Central      Rotina        60-79           8      True               0                            1                      0                    0                       0
1  EX-0138    PAC-0035  2026-08-02    M   19.0       Hemoglobina   12.500000    g/dL  Laboratório Central      Rotina        18-39           8     False               0                            0                      0                    0                       1
2  EX-0127    PAC-0165  2026-08-07    F   60.0 

## 9. Load e dicionário de dados

Exportar o dataset limpo para CSV e mostrar o dicionário de dados (coluna, tipo, descrição).

In [9]:
# Exportar o dataset limpo para CSV e mostrar o dicionário de dados (coluna, tipo, descrição).
exames_laboratoriais_limpo.to_csv( SAIDA / "exames_padronizados_final.csv", index=False)

# Mostrar o dicionário de dados
print("\nDicionário de dados:")
for col in exames_laboratoriais_limpo.columns:
    print(f"- {col} ({exames_laboratoriais_limpo[col].dtype}):\n{exames_laboratoriais_limpo[col].describe()}\n")

descricao = {
    "id_exame": "Identificador único do exame",
    "id_paciente": "Identificador único do paciente",
    "data_coleta": "Data da realização da coleta do exame",
    "sexo": "Sexo do paciente (F / M)",
    "idade": "Idade do paciente em anos",
    "tipo_exame": "Nome do exame realizado",
    "resultado": "Valor numérico do resultado do exame",
    "unidade": "Unidade de medida do resultado (ex: mg/dL, g/dL)",
    "laboratorio": "Nome da unidade laboratorial responsável",
    "prioridade": "Nível de prioridade do atendimento (Rotina, Urgente, Emergência)",
    "faixa_etaria": "Faixa etária do paciente '0-17', '18-39', '40-59', '60-79', '80+' (pd.cut)",
    "mes_coleta": "Mês da coleta (1-12)",
    "alterado": "Indicador se o resultado está fora da referência (True/False)",
    "prioridade_cod": "Prioridade ordinal (0=Rotina, 1=Urgente, 2=Emergência)",
    "tipo_exame_Colesterol total": "Codificação One-Hot para Colesterol total (0/1)",
    "tipo_exame_Creatinina": "Codificação One-Hot para Creatinina (0/1)",
    "tipo_exame_Glicemia": "Codificação One-Hot para Glicemia (0/1)",
    "tipo_exame_Hemoglobina": "Codificação One-Hot para Hemoglobina (0/1)"
}
dicionario = pd.DataFrame({"coluna": exames_laboratoriais_limpo.columns,
                           "tipo": exames_laboratoriais_limpo.dtypes.astype(str).values,
                           "descricao": [descricao.get(c, "one-hot do diagnóstico (0/1)") for c in exames_laboratoriais_limpo.columns]})
dicionario.to_csv(SAIDA / "dicionario_de_dados.csv", index=False, encoding="utf-8")
print(dicionario.to_string(index=False))


Dicionário de dados:
- id_exame (str):
count         150
unique        150
top       EX-0120
freq            1
Name: id_exame, dtype: object

- id_paciente (str):
count          150
unique         106
top       PAC-0178
freq             4
Name: id_paciente, dtype: object

- data_coleta (datetime64[us]):
count                    150
mean     2026-08-15 12:28:48
min      2026-07-02 00:00:00
25%      2026-07-24 06:00:00
50%      2026-08-14 12:00:00
75%      2026-09-07 00:00:00
max      2026-09-28 00:00:00
Name: data_coleta, dtype: object

- sexo (str):
count     150
unique      2
top         M
freq       80
Name: sexo, dtype: object

- idade (float64):
count    150.000000
mean      52.906667
std       20.513736
min       18.000000
25%       35.250000
50%       50.500000
75%       69.000000
max       91.000000
Name: idade, dtype: float64

- tipo_exame (str):
count                  150
unique                   4
top       Colesterol total
freq                    45
Name: tipo_exame, dtype:

## 10. Conclusão

Neste projeto, estruturamos um pipeline ETL completo para padronizar 150 exames laboratoriais, realizando a remoção de duplicatas, o tratamento de ausentes por mediana e a eliminação do código de erro 999. Desenvolvemos a engenharia de atributos com as variáveis faixa_etaria, mes_coleta e o indicador clínico alterado, aplicando também a codificação ordinal e one-hot. O dataset final foi validado por testes automatizados com assert para garantir total integridade e ausência de nulos antes da exportação final.

Essa limpeza transforma dados brutos e inconsistentes em um ativo pronto para alimentar modelos de Machine Learning e painéis de inteligência em saúde facilitado com a padronização de categorias e a criação de indicadores clínicos relevantes. A análise revelou que a maioria dos exames estava dentro dos limites de referência, mas alguns casos de resultados alterados foram identificados, destacando a importância da padronização e validação dos dados para decisões clínicas confiáveis.